# Training Data Format Comparison
### Why v0.1 → 0.57 and what the correct training target looks like

This notebook walks through the **four data formats** seen across training runs,  
using the same competition problem rendered in each, with colour-coding to show  
what's present, missing, or malformed at each stage.

| Run | Data source | Score |
|-----|-------------|-------|
| v0.1 | `train.csv` bare answer | 0.57 |
| v0.5 | template-reasoning SFT | 0.60 |
| v0.6 | GRPO from v0.5 (partial warm-start) | pending |
| v0.7 | GRPO from v0.5 (full warm-start, fixed) | — |

**Colour key used throughout:**
<span style='background:#c8f7c5;padding:2px 6px;border-radius:3px'>green = correct / present</span> &nbsp;
<span style='background:#ffd6d6;padding:2px 6px;border-radius:3px'>red = missing / wrong</span> &nbsp;
<span style='background:#fff3cd;padding:2px 6px;border-radius:3px'>yellow = partial / template only</span> &nbsp;
<span style='background:#d0e8ff;padding:2px 6px;border-radius:3px'>blue = \boxed{} answer</span>

In [1]:
import csv, json, re
from pathlib import Path
from IPython.display import display, HTML

WORKSPACE = Path("..")

# ── colour palette ────────────────────────────────────────────────────────────
C = {
    "green":  "#c8f7c5",
    "red":    "#ffd6d6",
    "yellow": "#fff3cd",
    "blue":   "#d0e8ff",
    "gray":   "#f0f0f0",
    "orange": "#ffe4b5",
    "purple": "#ead9ff",
}

def hl(text, colour, bold=False):
    """Wrap text in a coloured span."""
    weight = 'font-weight:bold;' if bold else ''
    return (f'<span style="background:{colour};{weight}'
            f'padding:1px 4px;border-radius:3px;font-family:monospace">{text}</span>')

def missing(label):
    return (f'<span style="background:{C["red"]};padding:1px 6px;border-radius:3px;'
            f'font-family:monospace;color:#900">❌ {label}</span>')

def present(label):
    return (f'<span style="background:{C["green"]};padding:1px 6px;border-radius:3px;'
            f'font-family:monospace;color:#060">✅ {label}</span>')

def partial(label):
    return (f'<span style="background:{C["yellow"]};padding:1px 6px;border-radius:3px;'
            f'font-family:monospace;color:#663">⚠️ {label}</span>')

def section(title, subtitle=""):
    sub = f'<div style="color:#555;margin-top:2px;font-size:0.9em">{subtitle}</div>' if subtitle else ''
    return HTML(f'<h3 style="margin-bottom:4px;border-left:4px solid #666;padding-left:10px">{title}</h3>{sub}')

def pre(content, title="", border_colour="#ccc"):
    """Render a monospace box with optional title."""
    t = f'<div style="font-size:0.8em;font-weight:bold;color:#555;margin-bottom:3px">{title}</div>' if title else ''
    return HTML(
        f'{t}<div style="background:#fafafa;border:1px solid {border_colour};'
        f'border-radius:6px;padding:14px 18px;font-family:monospace;'
        f'font-size:0.88em;line-height:1.7;white-space:pre-wrap">'
        f'{content}</div>'
    )

print("Helpers ready.")

Helpers ready.


In [2]:
# ── load data ─────────────────────────────────────────────────────────────────

# Build a lookup: prompt-prefix → {user, assistant, bucket} from v0.5 SFT
v05_by_prefix = {}
with open(WORKSPACE / "data/v0.5_train.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        msgs = rec["messages"]
        user = next(m["content"] for m in msgs if m["role"] == "user")
        asst = next(m["content"] for m in msgs if m["role"] == "assistant")
        v05_by_prefix[user[:60]] = {"user": user, "assistant": asst, "bucket": rec["bucket"]}

# Pick one example per problem type
WANT_BUCKETS = ["bit_like", "cipher_like", "numeral_like"]
examples = {}  # bucket → {id, prompt, answer, v05_user, v05_asst}

with open(WORKSPACE / "data/train.csv") as f:
    for row in csv.DictReader(f):
        key = row["prompt"][:60]
        if key in v05_by_prefix:
            bucket = v05_by_prefix[key]["bucket"]
            if bucket in WANT_BUCKETS and bucket not in examples:
                examples[bucket] = {
                    "id":      row["id"],
                    "prompt":  row["prompt"],
                    "answer":  row["answer"],
                    "v05_user": v05_by_prefix[key]["user"],
                    "v05_asst": v05_by_prefix[key]["assistant"],
                }
        if len(examples) == len(WANT_BUCKETS):
            break

for bucket, ex in examples.items():
    print(f"{bucket:15s}  id={ex['id']}  answer={ex['answer']!r}")
print("\nData loaded.")

bit_like         id=00066667  answer='10010111'
cipher_like      id=00189f6a  answer='cat imagines book'
numeral_like     id=001b24c4  answer='XXXVIII'

Data loaded.


---
## FORMAT 1 — v0.1 Raw Competition Data

Training source: `data/train.csv` — columns `prompt` + `answer`.

The model is trained with a supervised loss directly on the bare answer string.  
**No** `\boxed{}`. **No** reasoning. **No** `<think>` block.  

At inference the Nemotron-H chat template injects `<think>\n` before the assistant turn,  
so the model is forced into a thinking context it was **never trained to produce or close**.

In [3]:
for bucket, ex in examples.items():
    prompt_short = ex["prompt"][:350] + (" …" if len(ex["prompt"]) > 350 else "")
    answer = ex["answer"]

    content = (
        f"{hl('PROMPT', C['gray'], bold=True)}\n"
        f"{prompt_short}\n\n"
        f"{hl('TARGET (answer column — what SFT trains the model to output)', C['gray'], bold=True)}\n"
        f"{hl(answer, C['red'], bold=True)}\n\n"
        f"{missing('<think>')}  "
        f"{missing('</think>')}  "
        f"{missing(r'\\boxed{}') }  "
        f"{missing('reasoning trace')}\n\n"
        f"<span style='color:#900;font-size:0.9em'>Chat template prepends "
        f"{hl('<think>\\n', C['red'])} at inference — the model enters a thinking block "
        f"it was never trained to write or close.</span>"
    )
    display(section(f"v0.1 · {bucket} · id={ex['id']}"))
    display(pre(content, border_colour=C["red"]))
    print()

---
## FORMAT 2 — v0.5 Template-Reasoning SFT

Training source: `data/v0.5_train.jsonl` — messages format, user + assistant turns.

The assistant target is a **template sentence** (`"I identify one rule…"`) followed by `\boxed{answer}`.  
This is better than v0.1 — `\boxed{}` is present — but the training target **never includes  
`<think>` or `</think>`**. At inference, the model is still placed inside an open `<think>` block  
and has no training signal for how to close it before writing `\boxed{}`.

In [4]:
def highlight_v05_asst(text):
    """Highlight \\boxed{} in blue, leave template reasoning in yellow."""
    boxed_re = re.compile(r'(\\boxed\{[^}]*\})')
    parts = boxed_re.split(text)
    out = ""
    for p in parts:
        if boxed_re.match(p):
            out += hl(p, C["blue"], bold=True)
        elif p:
            out += hl(p, C["yellow"])
    return out

for bucket, ex in examples.items():
    user_short   = ex["v05_user"][:350] + (" …" if len(ex["v05_user"]) > 350 else "")
    asst_hl      = highlight_v05_asst(ex["v05_asst"])

    content = (
        f"{hl('USER', C['gray'], bold=True)}\n"
        f"{user_short}\n\n"
        f"{hl('ASSISTANT  (training target)', C['gray'], bold=True)}\n"
        f"{asst_hl}\n\n"
        f"{missing('<think>')}  "
        f"{missing('</think>')}  "
        f"{present(r'\\boxed{}')}  "
        f"{partial('template reasoning')}\n\n"
        f"<span style='color:#663;font-size:0.9em'>Model sees "
        f"{hl('<think>\\n', C['red'])} prepended by chat template at inference, "
        f"then outputs the template sentence with no "
        f"{hl('</think>', C['red'])} closure.</span>"
    )
    display(section(f"v0.5 · {bucket} · id={ex['id']}"))
    display(pre(content, border_colour=C["yellow"]))
    print()

---
## FORMAT 3 — kishanvavdara Full CoT Dataset (optional download)

Source: `kishanvavdara/nemotron-reasoning-traj` on Kaggle.  
Generated by running Nemotron-30B on every competition problem and keeping traces  
where `correctness == 'true'` (~8,800 rows from 9,500).

The CoT trace is a full reasoning chain — the right *content* — but the dataset rows  
still need a `<think>…</think>` wrapper applied when building training targets.

In [5]:
cot_rows = None
cot_by_id = {}

try:
    import kagglehub
    print("Downloading kishanvavdara/nemotron-reasoning-traj …")
    path = kagglehub.dataset_download("kishanvavdara/nemotron-reasoning-traj")
    csv_path = next(Path(path).glob("**/*.csv"), None)
    if csv_path:
        with open(csv_path, encoding="utf-8") as f:
            cot_rows = list(csv.DictReader(f))
        cot_by_id = {r["id"]: r for r in cot_rows if "id" in r}
        print(f"Loaded {len(cot_rows)} rows. Columns: {list(cot_rows[0].keys())}")
    else:
        print("No CSV found in download.")
except ImportError:
    print("kagglehub not installed — skipping.  pip install kagglehub")
except Exception as e:
    print(f"Download failed: {e}")

/home/msusol/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download failed: [Errno 13] Permission denied: '/home/msusol/.cache/kagglehub/datasets'


In [6]:
def highlight_cot(text, max_chars=600):
    """Highlight \\boxed{} in blue; rest is the reasoning trace (green)."""
    if not text:
        return hl("[no cot column in this dataset]", C["red"])
    preview = text[:max_chars] + (" …" if len(text) > max_chars else "")
    boxed_re = re.compile(r'(\\boxed\{[^}]*\})')
    parts = boxed_re.split(preview)
    out = ""
    for p in parts:
        if boxed_re.match(p):
            out += hl(p, C["blue"], bold=True)
        elif p:
            out += hl(p, C["green"])
    return out

if cot_rows is None:
    display(HTML(
        f'<div style="background:{C["gray"]};padding:14px;border-radius:6px">'
        f'<b>kagglehub not available.</b> Run the cell above after '
        f'<code>pip install kagglehub</code> and re-run this cell.</div>'
    ))
else:
    # Pick the CoT column name (varies by dataset version)
    sample = cot_rows[0]
    cot_col = next((c for c in ["cot","reasoning","response","completion","solution"] if c in sample), None)
    print(f"CoT column: {cot_col!r}")

    for bucket, ex in examples.items():
        row = cot_by_id.get(ex["id"])
        cot_text  = row.get(cot_col, "") if (row and cot_col) else ""
        correct   = row.get("correctness", "?") if row else "[not found]"
        cot_hl    = highlight_cot(cot_text)

        content = (
            f"{hl('PROMPT (same as v0.1)', C['gray'], bold=True)} — truncated\n"
            f"{ex['prompt'][:200]} …\n\n"
            f"{hl(f'CORRECTNESS: {correct}   ANSWER: {ex["answer"]}', C['gray'], bold=True)}\n\n"
            f"{hl('FULL CoT TRACE (first 600 chars)', C['gray'], bold=True)}\n"
            f"{cot_hl}\n\n"
            f"{missing('<think>')}  "
            f"{missing('</think>')}  "
            f"{present(r'\\boxed{}')}  "
            f"{present('full reasoning trace')}\n\n"
            f"<span style='color:#060;font-size:0.9em'>Needs '"
            f"{hl('<think>', C['green'])} … {hl('</think>', C['green'])} "
            f"wrapper applied to become a correct Nemotron-H training target.</span>"
        )
        display(section(f"CoT · {bucket} · id={ex['id']}"))
        display(pre(content, border_colour=C["green"]))
        print()

---
## FORMAT 4 — Correct Inference Target

What the model **must** produce at inference — and what training data **must** look like  
to teach this behaviour.

The Nemotron-H chat template auto-prepends `<think>\n` when `add_generation_prompt=True`,  
so the assistant turn in training data should begin with a reasoning trace and close  
with `</think>\n\boxed{answer}`.  The full sequence the tokenizer sees is shown below.

In [7]:
# Realistic short reasoning traces for each problem type (hand-crafted to show format)
EXAMPLE_REASONING = {
    "bit_like": (
        "Looking at the examples: 01010001 → 11011101, 00001001 → 01101101.\n"
        "Testing bit-reversal: reverse(01010001) = 10001010. No.\n"
        "Testing NOT: ~01010001 = 10101110. No.\n"
        "Testing rotate-left-1 then XOR 10101010:\n"
        "  rotate_left(01010001) = 10100010, XOR 10101010 = 00001000. No.\n"
        "Testing XOR with 10101010 then rotate-right-1 …\n"
        "[...continued reasoning...]\n"
        "Rule identified. Applying to target input."
    ),
    "cipher_like": (
        "Looking at the examples to find the mapping pattern.\n"
        "'ucoov pwgtfyoqg vorq yrjjoe' → 'queen discovers near valley'\n"
        "Checking letter shift: u(+?) = q → shift -4. p(+?) = d → shift -12. Not constant.\n"
        "Checking reverse-alphabet: u=6th from end → f. No.\n"
        "Checking word-level mapping: ucoov=queen, pwgtfyoqg=discovers…\n"
        "The cipher maps words wholesale. Extracting all word pairs…\n"
        "[...building lookup table from examples...]\n"
        "Applying lookup to input words."
    ),
    "numeral_like": (
        "Looking at examples: 11→XI, 15→XV, 94→XCIV, 19→XIX.\n"
        "This is standard Roman numeral conversion.\n"
        "38 = 30 + 8\n"
        "30 = XXX  (three tens)\n"
        "8  = VIII (five + three ones)\n"
        "38 = XXXVIII"
    ),
}

def correct_target_html(prompt, answer, reasoning):
    prompt_hl  = prompt[:300] + (" …" if len(prompt) > 300 else "")
    return (
        # ── chat template tokens ──────────────────────────────────────────
        f"{hl('<|im_start|>system', C['gray'])}\n"
        f"{hl('detailed thinking on', C['gray'])}{hl('<|im_end|>', C['gray'])}\n"
        f"{hl('<|im_start|>user', C['gray'])}\n"
        f"{prompt_hl}\n"
        f"Please put your final answer inside \\boxed{{}}.\n"
        f"{hl('<|im_end|>', C['gray'])}\n"
        f"{hl('<|im_start|>assistant', C['gray'])}\n"
        # ── assistant turn (the part the model must learn to generate) ─────
        f"{hl('<think>', C['green'], bold=True)}\n"
        f"{hl(reasoning, C['green'])}\n"
        f"{hl('</think>', C['green'], bold=True)}\n"
        f"{hl(f'\\boxed{{{answer}}}', C['blue'], bold=True)}\n\n"
        # ── checklist ──────────────────────────────────────────────────────
        f"{present('<think>')}  "
        f"{present('</think>')}  "
        f"{present(f'\\boxed{{}}')}  "
        f"{present('reasoning trace')}\n\n"
        f"<span style='color:#060;font-size:0.9em'>This is the only format "
        f"where every element is present. SFT on traces like this (from "
        f"kishanvavdara + <think> wrapper) teaches the model the full output structure.  "
        f"GRPO then reinforces correctness within this structure.</span>"
    )

for bucket, ex in examples.items():
    reasoning = EXAMPLE_REASONING.get(bucket, "[reasoning trace]")
    content   = correct_target_html(ex["prompt"], ex["answer"], reasoning)
    display(section(f"Correct target · {bucket} · id={ex['id']}"))
    display(pre(content, border_colour=C["green"]))
    print()

---
## Summary — What Each Format Gets Right

In [8]:
def check(ok, label=None):
    if ok == True:  return present(label or "✓")
    if ok == False: return missing(label or "✗")
    return partial(label or "~")

rows_html = ""
table_data = [
    #  Format name           score   <think>  </think>  \boxed{}  reasoning     GRPO-ready
    ("v0.1 raw answer",      "0.57",  False,   False,    False,    False,        False),
    ("v0.5 template SFT",    "0.60",  False,   False,    True,     None,         None),
    ("kishanvavdara CoT",    "—",     False,   False,    True,     True,         None),
    ("Correct target",       "—",     True,    True,     True,     True,         True),
]

header = (
    "<tr style='background:#f0f0f0'>" +
    "".join(f"<th style='padding:8px 14px;text-align:left'>{h}</th>" for h in
            ["Format", "Score", "&lt;think&gt;", "&lt;/think&gt;",
             r"\boxed{}", "Reasoning", "GRPO-ready"]) +
    "</tr>"
)

for name, score, think_open, think_close, boxed, reasoning, grpo in table_data:
    cells = [
        f"<td style='padding:8px 14px;font-weight:bold'>{name}</td>",
        f"<td style='padding:8px 14px;text-align:center'>{score}</td>",
        f"<td style='padding:8px 14px;text-align:center'>{check(think_open)}</td>",
        f"<td style='padding:8px 14px;text-align:center'>{check(think_close)}</td>",
        f"<td style='padding:8px 14px;text-align:center'>{check(boxed)}</td>",
        f"<td style='padding:8px 14px;text-align:center'>{check(reasoning)}</td>",
        f"<td style='padding:8px 14px;text-align:center'>{check(grpo)}</td>",
    ]
    rows_html += "<tr>" + "".join(cells) + "</tr>"

table = (
    "<table style='border-collapse:collapse;font-family:sans-serif;font-size:0.9em;width:100%'>"
    f"<thead>{header}</thead><tbody>{rows_html}</tbody></table>"
)
display(HTML(table))

Format,Score,<think>,</think>,\boxed{},Reasoning,GRPO-ready
v0.1 raw answer,0.57,❌ ✗,❌ ✗,❌ ✗,❌ ✗,❌ ✗
v0.5 template SFT,0.60,❌ ✗,❌ ✗,✅ ✓,⚠️ ~,⚠️ ~
kishanvavdara CoT,—,❌ ✗,❌ ✗,✅ ✓,✅ ✓,⚠️ ~
Correct target,—,✅ ✓,✅ ✓,✅ ✓,✅ ✓,✅ ✓


In [9]:
display(HTML(f"""
<div style='font-family:sans-serif;font-size:0.9em;line-height:1.8;
            background:{C['gray']};padding:16px 20px;border-radius:8px;margin-top:12px'>
<b>Key takeaways:</b><br>
• {hl('v0.1', C['red'])} — bare answer, no format. Model learns domain pattern but actively unlearns reasoning.
  At inference, chat template forces &lt;think&gt;\\n but model was never trained to write or close it.
  Result: degenerate or empty reasoning, raw answer spat out. Score: <b>0.57</b>.<br><br>
• {hl('v0.5', C['yellow'])} — adds \\boxed{{}} and a template sentence. Better, but no &lt;think&gt;/&lt;/think&gt; in
  training data. Model enters the &lt;think&gt; block at inference with no training signal for closing it.
  Achieves <b>0.60</b> because the format regression from v0.1 is partially corrected.<br><br>
• {hl('kishanvavdara CoT', C['orange'])} — full Nemotron-30B reasoning trace, correct content,
  but still needs &lt;think&gt;…&lt;/think&gt; wrapper applied before use as SFT targets.<br><br>
• {hl('Correct target', C['green'])} — &lt;think&gt;, reasoning, &lt;/think&gt;, \\boxed{{answer}} all present.
  SFT on this format teaches the model <i>exactly</i> what to generate at inference.
  This is the warm-start v0.7 GRPO needs to build on.
</div>
"""))